In [15]:
import os, warnings

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

warnings.filterwarnings("ignore", message=r"KMeans is known to have a memory leak on Windows with MKL.*")

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
import urllib.parse

import plotly.graph_objects as go

In [16]:
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

SCHEMA_NAME = "a1_fct_vision_testlog_txt_processing_history"
TABLE_NAME  = "fct_vision_testlog_txt_processing_history"

def get_engine(config=DB_CONFIG):
    user = config["user"]
    password = urllib.parse.quote_plus(config["password"])
    host = config["host"]
    port = config["port"]
    dbname = config["dbname"]
    conn_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
    return create_engine(conn_str)

engine = get_engine()

query = text(f"""
SELECT
    station,
    remark,
    end_day,
    end_time,
    result,
    goodorbad
FROM {SCHEMA_NAME}.{TABLE_NAME}
WHERE station IN ('Vision1','Vision2')
  AND COALESCE(result,'') <> 'FAIL'
  AND COALESCE(goodorbad,'') <> 'BadFile'
ORDER BY end_day ASC, end_time ASC
""")

df = pd.read_sql(query, engine)
df.head(), df.shape

(   station remark   end_day  end_time result goodorbad
 0  Vision1     PD  20251001  01:25:33   PASS  GoodFile
 1  Vision2     PD  20251001  01:25:44   PASS  GoodFile
 2  Vision1     PD  20251001  01:25:58   PASS  GoodFile
 3  Vision2     PD  20251001  01:25:59   PASS  GoodFile
 4  Vision1     PD  20251001  01:26:12   PASS  GoodFile,
 (101810, 6))

In [17]:
# --- datetime 생성
df["end_day"] = df["end_day"].astype(str).str.replace(r"\D", "", regex=True)
dt_str = df["end_day"] + " " + df["end_time"].astype(str).str.strip()
df["end_dt"] = pd.to_datetime(dt_str, errors="coerce", format="mixed")
df = df.dropna(subset=["end_dt"]).copy()

# --- 정렬(요구 4)
df = df.sort_values(["end_dt", "station", "remark"]).reset_index(drop=True)

# --- month 생성(요구 8-4)
df["month"] = df["end_dt"].dt.strftime("%Y%m")

# --- 연속 run id 생성
# station 값이 바뀌는 지점마다 run_id 증가
df["run_id"] = (df["station"] != df["station"].shift(1)).cumsum()

# run 길이 계산
run_sizes = df.groupby("run_id")["station"].size().rename("run_len")
df = df.merge(run_sizes, on="run_id", how="left")

# "Vision1만 연속" 또는 "Vision2만 연속"이 10개 이상인 run은 vision_only로 표시
df["is_vision_only_run"] = df["run_len"] >= 10

df[["end_dt","station","remark","run_id","run_len","is_vision_only_run"]].head(30)

,end_dt,station,remark,run_id,run_len,is_vision_only_run
0,2025-10-01 01:25:33,Vision1,PD,1,1,False
1,2025-10-01 01:25:44,Vision2,PD,2,1,False
2,2025-10-01 01:25:58,Vision1,PD,3,1,False
3,2025-10-01 01:25:59,Vision2,PD,4,1,False
4,2025-10-01 01:26:12,Vision1,PD,5,1,False
5,2025-10-01 01:26:24,Vision2,PD,6,1,False
6,2025-10-01 01:26:39,Vision1,PD,7,1,False
7,2025-10-01 01:26:41,Vision2,PD,8,1,False
8,2025-10-01 01:26:55,Vision1,PD,9,1,False
9,2025-10-01 01:26:59,Vision2,PD,10,1,False


In [18]:
# (station, remark)별 op_ct 계산
df = df.sort_values(["station", "remark", "end_dt"]).copy()
df["op_ct"] = df.groupby(["station", "remark"])["end_dt"].diff().dt.total_seconds()

# 첫 행은 NaN 유지
# 600초 초과 제외(요구 9) -> 분석용 df_an
df_an = df.dropna(subset=["op_ct"]).copy()
df_an = df_an[df_an["op_ct"] <= 600].copy()

df_an.head(10)

,station,remark,end_day,end_time,result,goodorbad,end_dt,month,run_id,run_len,is_vision_only_run,op_ct
14297,Vision1,Non-PD,20251010,00:00:20,PASS,GoodFile,2025-10-10 00:00:20,202510,11629,1,False,16.0
14300,Vision1,Non-PD,20251010,00:00:40,PASS,GoodFile,2025-10-10 00:00:40,202510,11631,1,False,20.0
14302,Vision1,Non-PD,20251010,00:00:55,PASS,GoodFile,2025-10-10 00:00:55,202510,11633,1,False,15.0
14304,Vision1,Non-PD,20251010,00:01:10,PASS,GoodFile,2025-10-10 00:01:10,202510,11635,1,False,15.0
16264,Vision1,Non-PD,20251010,14:48:04,PASS,GoodFile,2025-10-10 14:48:04,202510,13199,16,True,15.0
16266,Vision1,Non-PD,20251010,14:48:17,PASS,GoodFile,2025-10-10 14:48:17,202510,13201,1,False,13.0
16268,Vision1,Non-PD,20251010,14:48:32,PASS,GoodFile,2025-10-10 14:48:32,202510,13203,1,False,15.0
16270,Vision1,Non-PD,20251010,14:48:48,PASS,GoodFile,2025-10-10 14:48:48,202510,13205,1,False,16.0
16272,Vision1,Non-PD,20251010,14:49:03,PASS,GoodFile,2025-10-10 14:49:03,202510,13207,1,False,15.0
16275,Vision1,Non-PD,20251010,14:49:24,PASS,GoodFile,2025-10-10 14:49:24,202510,13209,1,False,21.0


In [19]:
def boxplot_stats(values: np.ndarray):
    """values: 1D array"""
    values = values.astype(float)
    q1 = float(np.percentile(values, 25))
    med = float(np.percentile(values, 50))
    q3 = float(np.percentile(values, 75))
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    inlier = values[(values >= lower) & (values <= upper)]
    del_out_mean = float(np.mean(inlier)) if len(inlier) else np.nan

    return {
        "q1": q1, "median": med, "q3": q3,
        "lower": float(lower), "upper": float(upper),
        "del_out_mean": del_out_mean,
        "sample_amount": int(len(values))
    }

def make_plotly_box_json(values: np.ndarray, title: str):
    # 시각화 출력은 하지 않지만, json으로 저장
    fig = go.Figure()
    fig.add_trace(go.Box(y=values.astype(float), boxpoints=False, name=title))
    fig.update_layout(title=title, showlegend=False)
    return fig.to_json()

def fmt_range(a: float, b: float):
    # 첨부 이미지처럼 "2.5~3.8" 문자열
    return f"{a:.2f}~{b:.2f}"

In [20]:
rows = []

def summarize_subset(df_subset: pd.DataFrame, station_label_map=None):
    """
    df_subset: 분석 대상 dataframe(op_ct<=600, NaN 제거 포함)
    station_label_map: station 라벨 변경용 dict
    """
    if station_label_map is not None:
        df_subset = df_subset.copy()
        df_subset["station_out"] = df_subset["station"].map(station_label_map).fillna(df_subset["station"])
    else:
        df_subset = df_subset.copy()
        df_subset["station_out"] = df_subset["station"]

    group_cols = ["station_out", "remark", "month"]
    for (st, rk, mo), g in df_subset.groupby(group_cols, sort=True):
        vals = g["op_ct"].astype(float).to_numpy()
        if len(vals) == 0:
            continue

        stats = boxplot_stats(vals)
        plotly_json = make_plotly_box_json(vals, title=f"{st}-{rk}-{mo}")

        # 요구 8-3: 소수점 3자리 반올림하여 2자리 표시
        # -> 저장값 자체를 2자리로 round 처리
        q1 = round(stats["q1"], 2)
        med = round(stats["median"], 2)
        q3 = round(stats["q3"], 2)
        lower = round(stats["lower"], 2)
        upper = round(stats["upper"], 2)
        del_out_mean = round(stats["del_out_mean"], 2) if not np.isnan(stats["del_out_mean"]) else np.nan

        rows.append({
            "station": st,
            "remark": rk,
            "month": mo,
            "sample_amount": stats["sample_amount"],
            "op_ct_lower_outlier": fmt_range(lower, q1),
            "q1": q1,
            "median": med,
            "q3": q3,
            "op_ct_upper_outlier": fmt_range(q3, upper),
            "del_out_op_ct_av": del_out_mean,
            "plotly_json": plotly_json
        })

# ---- (A) 정상 대상: vision_only_run 제외
df_normal = df_an[df_an["is_vision_only_run"] == False].copy()
summarize_subset(df_normal)

# ---- (B) Vision_only 대상: vision_only_run만
df_only = df_an[df_an["is_vision_only_run"] == True].copy()
summarize_subset(df_only, station_label_map={"Vision1":"Vision1_only", "Vision2":"Vision2_only"})

In [21]:
summary_df = pd.DataFrame(rows)

# 정렬
summary_df = summary_df.sort_values(["remark","station","month"]).reset_index(drop=True)

# id 컬럼 추가(첨부 이미지처럼)
summary_df.insert(0, "id", np.arange(1, len(summary_df) + 1))

# 첨부 이미지처럼 컬럼 순서 맞추기
summary_df = summary_df[
    ["id","station","remark","month","sample_amount",
     "op_ct_lower_outlier","q1","median","q3","op_ct_upper_outlier",
     "del_out_op_ct_av","plotly_json"]
]

summary_df

,id,station,remark,month,sample_amount,op_ct_lower_outlier,q1,median,q3,op_ct_upper_outlier,del_out_op_ct_av,plotly_json
0,1,Vision1,Non-PD,202510,16340,13.50~15.00,15.0,15.0,16.0,16.00~17.50,15.04,"{""data"":[{""boxpoints"":false,""name"":""Vision1-No..."
1,2,Vision1,Non-PD,202512,2060,13.50~15.00,15.0,15.0,16.0,16.00~17.50,15.06,"{""data"":[{""boxpoints"":false,""name"":""Vision1-No..."
2,3,Vision1_only,Non-PD,202510,516,13.50~15.00,15.0,15.0,16.0,16.00~17.50,15.05,"{""data"":[{""boxpoints"":false,""name"":""Vision1_on..."
3,4,Vision1_only,Non-PD,202512,19,12.50~14.00,14.0,14.0,15.0,15.00~16.50,14.39,"{""data"":[{""boxpoints"":false,""name"":""Vision1_on..."
4,5,Vision2,Non-PD,202510,16448,5.50~13.00,13.0,15.0,18.0,18.00~25.50,15.67,"{""data"":[{""boxpoints"":false,""name"":""Vision2-No..."
5,6,Vision2,Non-PD,202512,1919,2.50~13.00,13.0,17.0,20.0,20.00~30.50,16.57,"{""data"":[{""boxpoints"":false,""name"":""Vision2-No..."
6,7,Vision2_only,Non-PD,202510,1650,4.00~13.00,13.0,15.0,19.0,19.00~28.00,16.13,"{""data"":[{""boxpoints"":false,""name"":""Vision2_on..."
7,8,Vision2_only,Non-PD,202512,139,5.00~14.00,14.0,17.0,20.0,20.00~29.00,16.53,"{""data"":[{""boxpoints"":false,""name"":""Vision2_on..."
8,9,Vision1,PD,202510,27391,3.00~15.00,15.0,19.0,23.0,23.00~35.00,19.48,"{""data"":[{""boxpoints"":false,""name"":""Vision1-PD..."
9,10,Vision1,PD,202512,1952,1.50~15.00,15.0,19.0,24.0,24.00~37.50,19.65,"{""data"":[{""boxpoints"":false,""name"":""Vision1-PD..."


In [23]:
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values
import pandas as pd

# =========================
# 대상 스키마 / 테이블
# =========================
TARGET_SCHEMA = "e2_vision_ct"   # 소문자 고정
TARGET_TABLE  = "vision_op_ct"

def get_conn_pg(config=DB_CONFIG):
    return psycopg2.connect(
        host=config["host"],
        port=config["port"],
        dbname=config["dbname"],
        user=config["user"],
        password=config["password"],
    )

with get_conn_pg() as conn:
    with conn.cursor() as cur:

        # 1) 스키마 생성 (있으면 PASS)
        cur.execute(
            sql.SQL("CREATE SCHEMA IF NOT EXISTS {}")
            .format(sql.Identifier(TARGET_SCHEMA))
        )

        # 2) 테이블 생성 (있으면 PASS)
        cur.execute(
            sql.SQL("""
            CREATE TABLE IF NOT EXISTS {}.{} (
                station              TEXT NOT NULL,
                remark               TEXT NOT NULL,
                month                TEXT NOT NULL,
                sample_amount        INTEGER,
                op_ct_lower_outlier  TEXT,
                q1                   DOUBLE PRECISION,
                median               DOUBLE PRECISION,
                q3                   DOUBLE PRECISION,
                op_ct_upper_outlier  TEXT,
                del_out_op_ct_av     DOUBLE PRECISION,
                plotly_json          JSONB,
                inserted_at          TIMESTAMP DEFAULT NOW(),

                CONSTRAINT uq_vision_op_ct UNIQUE (station, remark, month)
            )
            """).format(
                sql.Identifier(TARGET_SCHEMA),
                sql.Identifier(TARGET_TABLE)
            )
        )

        conn.commit()

        # 3) INSERT (중복 키면 PASS)
        insert_sql = sql.SQL("""
            INSERT INTO {}.{} (
                station, remark, month,
                sample_amount,
                op_ct_lower_outlier,
                q1, median, q3,
                op_ct_upper_outlier,
                del_out_op_ct_av,
                plotly_json
            )
            VALUES %s
            ON CONFLICT (station, remark, month) DO NOTHING
        """).format(
            sql.Identifier(TARGET_SCHEMA),
            sql.Identifier(TARGET_TABLE)
        )

        records = [
            (
                r["station"],
                r["remark"],
                r["month"],
                int(r["sample_amount"]) if pd.notna(r["sample_amount"]) else None,
                r["op_ct_lower_outlier"],
                float(r["q1"]) if pd.notna(r["q1"]) else None,
                float(r["median"]) if pd.notna(r["median"]) else None,
                float(r["q3"]) if pd.notna(r["q3"]) else None,
                r["op_ct_upper_outlier"],
                float(r["del_out_op_ct_av"]) if pd.notna(r["del_out_op_ct_av"]) else None,
                r["plotly_json"],
            )
            for _, r in summary_df.iterrows()
        ]

        execute_values(
            cur,
            insert_sql.as_string(conn),
            records,
            template="(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s::jsonb)"
        )

        conn.commit()

print(f"[OK] Insert attempted: {len(summary_df)} rows into {TARGET_SCHEMA}.{TARGET_TABLE} (duplicates PASSED)")

[OK] Insert attempted: 14 rows into e2_vision_ct.vision_op_ct (duplicates PASSED)
